# Study 833 — Deflated Sharpe Ratio — the teardown

The expected-maximum-Sharpe asymptotics, the observed-vs-formula inflation curve, the moment-aware Deflated Sharpe Ratio, the in-sample→out-of-sample collapse, the costed timer, and the 40-seed null calibration + honest positive control. All offline, seed 833.

In [1]:
R = {'n_trials': 1000, 'n_days': 1260, 'ann_vol': 0.15, 'seed': 833, 'fingerprint': 'f7e4b81df8a2', 'mean_col_sharpe': 0.026, 'obs_max_sharpe': 1.251, 'exp_max_sharpe': 1.456, 'sr_std': 0.02725, 'sr_std_theory': 0.02818, 'winner_sharpe': 1.251, 'winner_sr0_ann': 1.456, 'winner_excess': -0.205, 'winner_dsr': 0.324, 'winner_naive_t': 2.8, 'is_sharpe': 1.769, 'oos_sharpe': -0.28, 'oos_t_nw': -0.47, 'timer1_gross': -1.649, 'timer1_net': -3.649, 'timer1_t': -0.98, 'timer5_net': -11.649, 'timer5_t': -3.13, 'curve_N': [2, 5, 10, 25, 50, 100, 250, 500, 1000], 'curve_obs': [0.237, 0.487, 0.705, 0.892, 1.007, 1.151, 1.277, 1.371, 1.467], 'curve_pred': [0.233, 0.534, 0.704, 0.894, 1.018, 1.132, 1.27, 1.366, 1.456], 'n_seeds': 40, 'cal_naive_fire': 40, 'cal_naive_rate': 1.0, 'cal_dsr_fire': 0, 'cal_dsr_rate': 0.0, 'cal_mean_dsr': 0.509, 'cal_mean_excess': 0.011, 'honest_true': 1.0, 'honest_realised': 1.014, 'honest_mean_dsr': 0.965, 'honest_fire': 33, 'honest_rate': 0.82, 'planted_true': 2.0, 'planted_dsr': 0.648}

## The headline — best of 1,000 EMPTY strategies

Every column has a *true* Sharpe of exactly 0; the winner is pure selection luck.

In [2]:
print(f"mean column Sharpe (truth) : {R['mean_col_sharpe']:+.3f}  (~0)")
print(f"observed MAX Sharpe        : {R['obs_max_sharpe']:+.3f}")
print(f"expected MAX under null SR0: {R['exp_max_sharpe']:+.3f}  (N=1000)")
print(f"cross-trial SR std sqrt(V) : {R['sr_std']:.5f}  (theory 1/sqrt(T-1) = {R['sr_std_theory']:.5f})")

mean column Sharpe (truth) : +0.026  (~0)
observed MAX Sharpe        : +1.251
expected MAX under null SR0: +1.456  (N=1000)
cross-trial SR std sqrt(V) : 0.02725  (theory 1/sqrt(T-1) = 0.02818)


## The deflation — DSR of the winner

DSR = Φ((SR − SR0)·√(T−1) / √(1 − g₃·SR + (g₄−1)/4·SR²)). Below 0.95 ⇒ consistent with luck.

In [3]:
print(f"winner Sharpe {R['winner_sharpe']:+.2f}  vs expected-max bar SR0 {R['winner_sr0_ann']:+.2f}")
print(f"deflated EXCESS Sharpe (SR-SR0): {R['winner_excess']:+.2f}  (~0)")
print(f"DSR = {R['winner_dsr']:.3f}   but naive one-sample t = {R['winner_naive_t']:+.2f} (fools you)")

winner Sharpe +1.25  vs expected-max bar SR0 +1.46
deflated EXCESS Sharpe (SR-SR0): -0.20  (~0)
DSR = 0.324   but naive one-sample t = +2.80 (fools you)


## The inflation curve — E[max] vs N (observed, 40 seeds/point, vs formula)

In [4]:
for N, o, p in zip(R['curve_N'], R['curve_obs'], R['curve_pred']):
    print(f"N={N:>5}: observed best {o:+.3f}   formula E[max] {p:+.3f}")

N=    2: observed best +0.237   formula E[max] +0.233
N=    5: observed best +0.487   formula E[max] +0.534
N=   10: observed best +0.705   formula E[max] +0.704
N=   25: observed best +0.892   formula E[max] +0.894
N=   50: observed best +1.007   formula E[max] +1.018
N=  100: observed best +1.151   formula E[max] +1.132
N=  250: observed best +1.277   formula E[max] +1.270
N=  500: observed best +1.371   formula E[max] +1.366
N= 1000: observed best +1.467   formula E[max] +1.456


## Out-of-sample collapse + the costed timer (the Mirage)

In [5]:
print(f"in-sample Sharpe   {R['is_sharpe']:+.2f}  ->  out-of-sample {R['oos_sharpe']:+.2f} "
      f"(NW t = {R['oos_t_nw']:+.2f})")
print(f"timer @1 bp: gross {R['timer1_gross']:+.2f} -> net {R['timer1_net']:+.2f} bps/day (t={R['timer1_t']:+.2f})")
print(f"timer @5 bp:                 net {R['timer5_net']:+.2f} bps/day (t={R['timer5_t']:+.2f})")

in-sample Sharpe   +1.77  ->  out-of-sample -0.28 (NW t = -0.47)
timer @1 bp: gross -1.65 -> net -3.65 bps/day (t=-0.98)
timer @5 bp:                 net -11.65 bps/day (t=-3.13)


## Live control — the machinery is calibrated

A small live run: the naive screen fires on the null winner; the DSR does not; and an honest single strategy keeps a high DSR. (Fast: 12 null pools + 12 honest streams.)

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
from deflated_sharpe import strategy as st
cal = st.null_dsr_calibration(n_trials=1000, n_days=1260, ann_vol=0.15, n_seeds=12, base_seed=833)
hc  = st.honest_control(true_ann_sharpe=1.0, n_days=1260, ann_vol=0.15, n_seeds=12, base_seed=833)
print(f"NULL pools: naive |t|>=2 fires {cal['naive_fire']}/{cal['n_seeds']}  "
      f"vs DSR>=0.95 fires {cal['dsr_fire']}/{cal['n_seeds']}; mean DSR {cal['mean_dsr']:.3f} (~0.5)")
print(f"HONEST single strategy: mean DSR {hc['mean_dsr']:.3f}, DSR>=0.95 in "
      f"{hc['dsr_fire']}/{hc['n_seeds']}  -> the correction spares real skill")

NULL pools: naive |t|>=2 fires 12/12  vs DSR>=0.95 fires 0/12; mean DSR 0.517 (~0.5)
HONEST single strategy: mean DSR 0.940, DSR>=0.95 in 8/12  -> the correction spares real skill


## Verdict

- **Signal — None.** By construction: the tape is a certified null. The best of 1,000 empty strategies looks like a Sharpe-1.25 winner (naive *t* = +2.80) and is provably nothing.
- **Tradability — Mirage.** In-sample +1.77 → out-of-sample -0.28 (NW *t* = -0.47); net -3.65 bps/day at 1 bp. Nothing to harvest.
- **Does the trial count inflate the best Sharpe? — Confirmed.** Observed max tracks E[max] from N=2→1,000; the DSR shrinks the winner to a coin flip (mean 0.509, deflated excess ≈ +0.01), firing on 0/40 nulls vs 40/40 for the naive screen — while sparing the honest strategy (mean DSR 0.96). *(The synthetic controls prove the machinery is calibrated — never cited to support a real-tape stamp; there is no real tape.)*